# SyntecxHub Internship Program
## Domain: Machine Learning | Task: Week 1
### Project 1: House Price Prediction using Linear Regression

---
### Project Constraints & Objectives:
1. **Load a housing dataset, clean and explore features.**
2. **Select features, split train/test (80/20) and train a Linear Regression model.**
3. **Evaluate using RMSE and $R^2$, and interpret coefficients.**
4. **Save model and show example predictions.**

**Author:** Intern @ SyntecxHub  
**GitHub Repository:** `Syntecxhub_House_Price_Prediction`

## 1. Environment Setup & Dependency Imports

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

# Set visual style
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (9, 5)
print("All libraries successfully imported!")

## 2. Load Housing Dataset
We load the housing dataset containing real-world housing attributes including living area, bedroom/bathroom counts, build year, garage capacity, lot size, neighborhood quality ratings, and sale prices.

In [ ]:
data_path = os.path.join("..", "data", "housing_data.csv")
df = pd.read_csv(data_path)
print(f"Dataset Shape: {df.shape}")
df.head()

## 3. Data Cleaning & Inspection
We inspect missing values, handle duplicated records, and impute missing fields using domain-relevant strategies (median for area, mode for garage capacity).

In [ ]:
print("--- Missing Values Check ---")
print(df.isnull().sum())

# Remove duplicates
initial_count = len(df)
df.drop_duplicates(inplace=True)
print(f"\nDropped {initial_count - len(df)} duplicate records.")

# Impute missing values
df["Lot_Area"] = df["Lot_Area"].fillna(df["Lot_Area"].median())
df["Garage_Cars"] = df["Garage_Cars"].fillna(df["Garage_Cars"].mode()[0])
df["Neighborhood_Score"] = df["Neighborhood_Score"].fillna(df["Neighborhood_Score"].median())

print("\nMissing values after imputation:")
print(df.isnull().sum())

## 4. Feature Engineering
- **House_Age**: Calculated as `2024 - Year_Built` to capture the real-estate depreciation lifecycle.
- **Total_Rooms**: Aggregate of bedrooms and bathrooms.

In [ ]:
CURRENT_YEAR = 2024
df["House_Age"] = CURRENT_YEAR - df["Year_Built"]
df["House_Age"] = df["House_Age"].clip(lower=0)
df["Total_Rooms"] = df["Bedrooms"] + df["Bathrooms"]

df[["Year_Built", "House_Age", "Bedrooms", "Bathrooms", "Total_Rooms"]].head()

## 5. Exploratory Data Analysis (EDA)
Examining statistical distributions and correlation with target variable `Price`.

In [ ]:
# Descriptive Statistics
df.describe().round(2)

In [ ]:
# Correlation Matrix
numeric_df = df.select_dtypes(include=[np.number])
corr = numeric_df.corr()

plt.figure(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Feature Correlation Matrix", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# Price Distribution
plt.figure(figsize=(8, 5))
sns.histplot(df["Price"], kde=True, color="#2563eb", bins=30)
plt.title("Distribution of House Sale Prices", fontsize=13, fontweight="bold")
plt.xlabel("Price ($ USD)")
plt.ylabel("Count")
plt.show()

## 6. Feature Selection & Train / Test Split
We select the primary continuous and ordinal features driving home valuations and split 80% for training and 20% for testing.

In [ ]:
FEATURES = [
    'Square_Feet',
    'Bedrooms',
    'Bathrooms',
    'House_Age',
    'Garage_Cars',
    'Lot_Area',
    'Overall_Quality',
    'Neighborhood_Score'
]
TARGET = 'Price'

X = df[FEATURES]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)
print(f"Training Set: {X_train.shape[0]} rows")
print(f"Testing Set : {X_test.shape[0]} rows")

## 7. Model Training: Ordinary Least Squares Linear Regression

In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)
print("Linear Regression model successfully trained!")

## 8. Model Evaluation: RMSE & R²

In [ ]:
y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)

train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))

train_r2 = r2_score(y_train, y_train_pred)
test_r2 = r2_score(y_test, y_test_pred)

train_mae = mean_absolute_error(y_train, y_train_pred)
test_mae = mean_absolute_error(y_test, y_test_pred)

eval_df = pd.DataFrame({
    'Metric': ['R-squared (R²)', 'RMSE ($ USD)', 'MAE ($ USD)'],
    'Train Set': [f"{train_r2:.4f}", f"${train_rmse:,.2f}", f"${train_mae:,.2f}"],
    'Test Set': [f"{test_r2:.4f}", f"${test_rmse:,.2f}", f"${test_mae:,.2f}"]
})
eval_df

In [ ]:
# Visualizing Actual vs. Predicted on Test Set
plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_test_pred, alpha=0.55, color="#2563eb", label="Test Samples")
min_v = min(y_test.min(), y_test_pred.min())
max_v = max(y_test.max(), y_test_pred.max())
plt.plot([min_v, max_v], [min_v, max_v], color="#dc2626", linestyle="--", linewidth=2, label="Perfect Fit (y=x)")
plt.title(f"Actual vs. Predicted Prices (Test R² = {test_r2:.3f})", fontsize=13, fontweight="bold")
plt.xlabel("Actual Price ($ USD)")
plt.ylabel("Predicted Price ($ USD)")
plt.legend()
plt.show()

## 9. Coefficient Interpretation
Interpreting the model weights in real-world dollar impact:
- $\beta_0$ (Intercept): Baseline home value.
- $\beta_j$ (Slopes): Marginal increase in price per unit increase in feature $j$, holding all other features constant.

In [ ]:
print(f"Intercept (β₀): ${model.intercept_:,.2f}\n")
coeff_df = pd.DataFrame({
    'Feature': FEATURES,
    'Coefficient (β)': model.coef_,
    'Dollar Impact': [f"{'+' if c >= 0 else '-'}${abs(c):,.2f}" for c in model.coef_],
    'Interpretation': [
        f"Each +1 unit changes price by {'+' if c>=0 else '-'}${abs(c):,.2f}"
        for c in model.coef_
    ]
}).sort_values('Coefficient (β)', ascending=False)
coeff_df

## 10. Save Model Artifacts
Serializing the trained model to disk using `joblib` for deployment.

In [ ]:
model_dir = os.path.join("..", "models")
os.makedirs(model_dir, exist_ok=True)
model_path = os.path.join(model_dir, "house_price_model.pkl")
joblib.dump(model, model_path)
print(f"Model saved successfully to: {model_path}")

## 11. Example Predictions on Novel Unseen Properties
Testing our saved model on four distinct real-world housing archetypes.

In [ ]:
# Load saved model to verify persistence
loaded_model = joblib.load(model_path)

examples = pd.DataFrame([
    {'Square_Feet': 1100, 'Bedrooms': 2, 'Bathrooms': 1.0, 'House_Age': 20, 'Garage_Cars': 1, 'Lot_Area': 4500, 'Overall_Quality': 5, 'Neighborhood_Score': 6.0},
    {'Square_Feet': 2250, 'Bedrooms': 4, 'Bathrooms': 2.5, 'House_Age': 12, 'Garage_Cars': 2, 'Lot_Area': 9500, 'Overall_Quality': 7, 'Neighborhood_Score': 7.5},
    {'Square_Feet': 3650, 'Bedrooms': 5, 'Bathrooms': 4.0, 'House_Age': 2,  'Garage_Cars': 3, 'Lot_Area': 16000, 'Overall_Quality': 9, 'Neighborhood_Score': 9.2},
    {'Square_Feet': 1600, 'Bedrooms': 3, 'Bathrooms': 1.5, 'House_Age': 58, 'Garage_Cars': 1, 'Lot_Area': 6200, 'Overall_Quality': 6, 'Neighborhood_Score': 6.8}
])

archetype_names = [
    '1. Starter Home',
    '2. Suburban Family',
    '3. Modern Luxury Villa',
    '4. Vintage Cottage'
]

predictions = loaded_model.predict(examples[FEATURES])

results_df = examples.copy()
results_df.insert(0, 'Archetype', archetype_names)
results_df['Predicted_Price'] = [f"${p:,.2f}" for p in predictions]
results_df